# Data Loading

In [15]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [16]:
import os
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader, random_split
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt

# 1. Setup Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cpu


In [18]:
# 2. Fast Data Loading
import os
# Check the "MyDrive" vs "My Drive" naming convention
path_options = ['/content/drive/MyDrive/TrainingDataset', '/content/drive/My Drive/TrainingDataset']

for p in path_options:
    if os.path.exists(p):
        print(f"✅ Found it at: {p}")
        print(f"Contents: {os.listdir(p)}")
        break
else:
    print("❌ Folder not found. Try the 'Side Panel' method below.")

data_dir = "/content/drive/My Drive/TrainingDataset" # Update this if your folder name is different
spectra_data = np.load(os.path.join(data_dir, 'spectra_compiled.npy'))
geometry_data = np.load(os.path.join(data_dir, 'geometry_compiled.npy'))

# 3. Proper Scaling
# Global Normalization for Spectra
spec_min, spec_max = spectra_data.min(), spectra_data.max()
spectra_normalized = (spectra_data - spec_min) / (spec_max - spec_min)

# Standard Scaling for Geometry
scaler_geom = StandardScaler()
geometry_scaled = scaler_geom.fit_transform(geometry_data)

# 4. PyTorch DataLoaders
spectra_tensor = torch.tensor(spectra_normalized, dtype=torch.float32)
geometry_tensor = torch.tensor(geometry_scaled, dtype=torch.float32)

# When we load a batch, it will yield: (batch_spectra, batch_geometry)
dataset = TensorDataset(spectra_tensor, geometry_tensor)
train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size
train_dataset, val_dataset = random_split(dataset, [train_size, val_size])

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False)

print("Data loaded!")

❌ Folder not found. Try the 'Side Panel' method below.


FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/My Drive/TrainingDataset/spectra_compiled.npy'

# ML Model

In [ ]:
# --- 1. FORWARD MODEL (Geometry -> Spectrum) ---
class ForwardMLP(nn.Module):
    def __init__(self):
        super(ForwardMLP, self).__init__()
        self.net = nn.Sequential(
            nn.Linear(8, 64), nn.ReLU(),
            nn.Linear(64, 128), nn.ReLU(),
            nn.Linear(128, 256), nn.ReLU(),
            nn.Linear(256, 100) # Outputs a 100-point spectrum
        )
    def forward(self, x):
        return self.net(x)

# --- 2. INVERSE MODEL (Spectrum -> Geometry) ---
class InverseCNN(nn.Module):
    def __init__(self):
        super(InverseCNN, self).__init__()
        self.conv1 = nn.Conv1d(1, 16, kernel_size=5, stride=1, padding=2)
        self.conv2 = nn.Conv1d(16, 32, kernel_size=5, stride=2, padding=2)
        self.conv3 = nn.Conv1d(32, 64, kernel_size=5, stride=2, padding=2)

        self.fc1 = nn.Linear(64 * 25, 128)
        self.fc2 = nn.Linear(128, 8) # Outputs 8 geometry parameters
        self.relu = nn.ReLU()

    def forward(self, x):
        x = x.unsqueeze(1) # Reshape 100 points into a 1-channel signal
        x = self.relu(self.conv1(x))
        x = self.relu(self.conv2(x))
        x = self.relu(self.conv3(x))
        x = x.view(x.size(0), -1)
        x = self.relu(self.fc1(x))
        return self.fc2(x)

forward_model = ForwardMLP().to(device)
inverse_model = InverseCNN().to(device)
print("Models defined and moved to device!")

# Training of Froward Model

In [ ]:
import matplotlib.pyplot as plt

criterion = nn.MSELoss()
optimizer_fwd = optim.Adam(forward_model.parameters(), lr=0.001)

fwd_epochs = 100
fwd_train_losses = []
fwd_val_losses = []

print("Training Forward Model (Geometry -> Spectrum)...")
for epoch in range(fwd_epochs):
    # --- TRAINING ---
    forward_model.train()
    running_train_loss = 0.0
    for batch_spectra, batch_geometry in train_loader:
        batch_spectra, batch_geometry = batch_spectra.to(device), batch_geometry.to(device)

        pred_spectra = forward_model(batch_geometry)
        loss = criterion(pred_spectra, batch_spectra)

        optimizer_fwd.zero_grad()
        loss.backward()
        optimizer_fwd.step()
        running_train_loss += loss.item()

    avg_train_loss = running_train_loss / len(train_loader)
    fwd_train_losses.append(avg_train_loss)

    # --- VALIDATION ---
    forward_model.eval()
    running_val_loss = 0.0
    with torch.no_grad():
        for val_spectra, val_geometry in val_loader:
            val_spectra, val_geometry = val_spectra.to(device), val_geometry.to(device)
            val_pred_spectra = forward_model(val_geometry)
            val_loss = criterion(val_pred_spectra, val_spectra)
            running_val_loss += val_loss.item()

    avg_val_loss = running_val_loss / len(val_loader)
    fwd_val_losses.append(avg_val_loss)

    if (epoch + 1) % 10 == 0:
        print(f"Epoch [{epoch+1}/{fwd_epochs}], Train Loss: {avg_train_loss:.4f}, Val Loss: {avg_val_loss:.4f}")

# --- FREEZE THE MODEL ---
for param in forward_model.parameters():
    param.requires_grad = False
forward_model.eval()
print("\nForward Model trained and frozen!")

# --- PLOT THE FORWARD LOSS ---
plt.figure(figsize=(10, 5))
plt.plot(fwd_train_losses, label='Forward Train Loss', color='green', linewidth=2)
plt.plot(fwd_val_losses, label='Forward Val Loss', color='purple', linewidth=2)
plt.xlabel('Epochs', fontsize=12)
plt.ylabel('Mean Squared Error', fontsize=12)
plt.title('Forward Model Learning Curve', fontsize=14)
plt.legend(fontsize=12)
plt.grid(True, linestyle='--', alpha=0.7)
plt.yscale('log')
plt.show()

In [ ]:
# Grab one batch of validation data
batch_spectra, batch_geometry = next(iter(val_loader))

# Take the first sample
true_spectrum = batch_spectra[0].unsqueeze(0).to(device)
true_geometry = batch_geometry[0].unsqueeze(0).to(device)

# 1. Forward Model makes a prediction
with torch.no_grad():
    predicted_spectrum = forward_model(true_geometry)

# Move back to CPU for plotting
true_spectrum_np = true_spectrum.cpu().numpy().squeeze()
predicted_spectrum_np = predicted_spectrum.cpu().numpy().squeeze()

# --- PLOT THE SPECTRA ---
plt.figure(figsize=(10, 6))
plt.plot(true_spectrum_np, label='True Spectrum (Lumerical)', color='black', linewidth=3)
plt.plot(predicted_spectrum_np, label='Predicted Spectrum (Forward Model)', color='limegreen', linestyle='--', linewidth=2)
plt.xlabel('Wavelength Index (0 to 100)', fontsize=12)
plt.ylabel('Normalized Intensity', fontsize=12)
plt.title('Forward Model Validation: True vs. Predicted Spectrum', fontsize=14)
plt.legend(fontsize=12)
plt.grid(True, alpha=0.5)
plt.show()

# Training of Inverse Model

In [ ]:
import matplotlib.pyplot as plt

optimizer_inv = optim.Adam(inverse_model.parameters(), lr=0.001)
epochs = 100

tandem_train_losses = []
tandem_val_losses = []

print("Starting Tandem Training (Solving the Inverse Problem)...")
for epoch in range(epochs):
    # --- TRAINING ---
    inverse_model.train()
    running_train_loss = 0.0
    for batch_spectra, _ in train_loader:
        batch_spectra = batch_spectra.to(device)

        guessed_geometry = inverse_model(batch_spectra)
        produced_spectra = forward_model(guessed_geometry)
        loss = criterion(produced_spectra, batch_spectra)

        optimizer_inv.zero_grad()
        loss.backward()
        optimizer_inv.step()
        running_train_loss += loss.item()

    avg_train_loss = running_train_loss / len(train_loader)
    tandem_train_losses.append(avg_train_loss)

    # --- VALIDATION ---
    inverse_model.eval()
    running_val_loss = 0.0
    with torch.no_grad():
        for val_spectra, _ in val_loader:
            val_spectra = val_spectra.to(device)
            val_guessed_geometry = inverse_model(val_spectra)
            val_produced_spectra = forward_model(val_guessed_geometry)
            val_loss = criterion(val_produced_spectra, val_spectra)
            running_val_loss += val_loss.item()

    avg_val_loss = running_val_loss / len(val_loader)
    tandem_val_losses.append(avg_val_loss)

    if (epoch + 1) % 10 == 0:
        print(f"Epoch [{epoch+1}/{epochs}], Train Loss: {avg_train_loss:.4f}, Val Loss: {avg_val_loss:.4f}")

# --- PLOT THE LOSS ---
plt.figure(figsize=(10, 6))
plt.plot(tandem_train_losses, label='Tandem Train Loss', color='blue', linewidth=2)
plt.plot(tandem_val_losses, label='Tandem Val Loss', color='orange', linewidth=2)
plt.xlabel('Epochs', fontsize=12)
plt.ylabel('Mean Squared Error (Spectra Matching)', fontsize=12)
plt.title('Tandem Network Training Curve', fontsize=14)
plt.legend(fontsize=12)
plt.grid(True, linestyle='--', alpha=0.7)
plt.yscale('log')
plt.show()

In [ ]:
# Grab one batch of validation data
inverse_model.eval()
forward_model.eval()
batch_spectra, batch_geometry = next(iter(val_loader))

# Take the first sample from the batch
target_spectrum = batch_spectra[0].unsqueeze(0).to(device)
true_geometry_scaled = batch_geometry[0].unsqueeze(0).numpy()

# 1. Model makes a prediction
with torch.no_grad():
    predicted_geometry_scaled = inverse_model(target_spectrum)
    produced_spectrum = forward_model(predicted_geometry_scaled)

# Move back to CPU for plotting/printing
target_spectrum_np = target_spectrum.cpu().numpy().squeeze()
produced_spectrum_np = produced_spectrum.cpu().numpy().squeeze()

# 2. Un-scale the geometry to real physical units
true_geom_physical = scaler_geom.inverse_transform(true_geometry_scaled)[0]
pred_geom_physical = scaler_geom.inverse_transform(predicted_geometry_scaled.cpu().numpy())[0]

# --- PRINT PHYSICAL PARAMETERS ---
print("PHYSICAL GEOMETRY COMPARISON")
print("-" * 65)
print(f"{'Parameter':<18} | {'Ground Truth':<20} | {'Model Prediction':<20}")
print("-" * 65)
for i in range(4):
    print(f"Layer {i+1} Radius (m) | {true_geom_physical[i*2]:>20.4e} | {pred_geom_physical[i*2]:>20.4e}")
    print(f"Layer {i+1} Theta (rad)| {true_geom_physical[i*2 + 1]:>20.4f} | {pred_geom_physical[i*2 + 1]:>20.4f}")
print("-" * 65)
print("\n*Note: If parameters differ but spectra match perfectly, you have successfully witnessed physics degeneracy!*\n")

# --- PLOT THE SPECTRA ---
plt.figure(figsize=(10, 6))
# We can un-normalize the spectra for the plot if we want, but plotting the 0-1 scale is fine for visual check
plt.plot(target_spectrum_np, label='Target Spectrum (Ground Truth)', color='black', linewidth=3)
plt.plot(produced_spectrum_np, label='Produced Spectrum (Model Output)', color='red', linestyle='--', linewidth=2)
plt.xlabel('Wavelength Index (0 to 100)', fontsize=12)
plt.ylabel('Normalized Intensity', fontsize=12)
plt.title('Inverse Design Validation: Target vs. Achieved Spectrum', fontsize=14)
plt.legend(fontsize=12)
plt.grid(True, alpha=0.5)
plt.show()

# Saving the Model

In [ ]:
import joblib
import os

# 1. Define the save paths in your Google Drive folder
forward_save_path = os.path.join(data_dir, 'forward_model_mlp.pth')
inverse_save_path = os.path.join(data_dir, 'inverse_model_tandem.pth')

# 2. Save the learned weights (state_dicts)
torch.save(forward_model.state_dict(), forward_save_path)
torch.save(inverse_model.state_dict(), inverse_save_path)

# 2. Save the Geometry Scaler
scaler_save_path = os.path.join(data_dir, 'scaler_geom.pkl')
joblib.dump(scaler_geom, scaler_save_path)

# 3. Save the Spectra Min/Max values
limits_save_path = os.path.join(data_dir, 'spec_min_max.npy')
np.save(limits_save_path, np.array([spec_min, spec_max]))

print("Success! Inverse Model, Scaler, and Min/Max limits safely backed up.")

# Export Current Predictions for Simulation

In [ ]:
export_path = os.path.join(data_dir, 'lumerical_test_parameters.csv')

# Notice the extra brackets around [pred_geom_physical]!
np.savetxt(export_path, [pred_geom_physical], delimiter=",",
           header="Radius1,Theta1,Radius2,Theta2,Radius3,Theta3,Radius4,Theta4",
           comments='')

print(f"Parameters properly formatted and exported for Lumerical: {export_path}")

# Upload Model

In [ ]:
import torch
import joblib
import numpy as np
import os

data_dir = '/content/drive/MyDrive/PolaritonicONN/TrainingDataset'

# 1. Setup Device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# ---------------------------------------------------------
# 2. Load the Scaling Data
# ---------------------------------------------------------
limits_path = os.path.join(data_dir, 'spec_min_max.npy')
spec_limits = np.load(limits_path)
spec_min, spec_max = spec_limits[0], spec_limits[1]
print(f"Loaded Spectral Limits: Min = {spec_min:.4f}, Max = {spec_max:.4f}")

scaler_path = os.path.join(data_dir, 'scaler_geom.pkl')
scaler_geom = joblib.load(scaler_path)
print("Loaded Geometry Scaler.")

# ---------------------------------------------------------
# 3. Initialize Blank Inverse Model and Load Weights
# ---------------------------------------------------------
# Make sure you have run the cell containing `class InverseCNN(nn.Module):` first!
inverse_model = InverseCNN().to(device)

inverse_path = os.path.join(data_dir, 'inverse_model_tandem.pth')

# Inject the saved weights into the empty model
inverse_model.load_state_dict(torch.load(inverse_path, map_location=device, weights_only=True))

# ---------------------------------------------------------
# 4. Lock the Model for Prediction
# ---------------------------------------------------------
inverse_model.eval()

print("\nSuccess! The Inverse Model is fully loaded and ready to design devices.")

In [ ]:
import torch
import os

# 1. Instantiate the exact same blueprint you used when saving
# (Make sure the ForwardMLP class is defined in your current session!)
forward_model = ForwardMLP().to(device)

# 2. Define the exact path to your saved file
# Update this filename if you saved it as something else!
forward_save_path = os.path.join(data_dir, 'forward_model_mlp.pth')

# 3. Load the weights (state_dict) into the empty model
# map_location=device ensures it loads correctly whether you are on CPU or GPU
forward_model.load_state_dict(torch.load(forward_save_path, map_location=device))

# 4. CRITICAL: Lock the model for inference
forward_model.eval()

print(f"Success! Forward model loaded from: {forward_save_path}")
print("Ready for predictions!")

# Use Inverse Model to Predict Geometry based on Targeted Spectra

In [ ]:
import numpy as np
import torch
import matplotlib.pyplot as plt
import pandas as pd
import os
from scipy.interpolate import interp1d

def design_from_target_and_export(raw_36_points, spec_min, spec_max, export_dir):
    """
    Squashes a target to the training data's global min/max, predicts the
    physical geometry, and exports those parameters directly to a CSV.
    """
    # ---------------------------------------------------------
    # 1. Interpolate and Squash
    # ---------------------------------------------------------
    x_original = np.linspace(0, 1, len(raw_36_points))
    x_target = np.linspace(0, 1, 100)

    interpolator = interp1d(x_original, raw_36_points, kind='cubic')
    target_100_pts = interpolator(x_target)

    target_min = np.min(target_100_pts)
    target_max = np.max(target_100_pts)

    squashed_spectrum = spec_min + ((target_100_pts - target_min) / (target_max - target_min)) * (spec_max - spec_min)

    # ---------------------------------------------------------
    # 2. Normalize for AI and Predict
    # ---------------------------------------------------------
    ai_ready_spectrum = (squashed_spectrum - spec_min) / (spec_max - spec_min)
    spectrum_tensor = torch.tensor(ai_ready_spectrum, dtype=torch.float32).unsqueeze(0).to(device)

    with torch.no_grad():
        pred_scaled_geom = inverse_model(spectrum_tensor)

    pred_physical_geom = scaler_geom.inverse_transform(pred_scaled_geom.cpu().numpy())[0]

    # ---------------------------------------------------------
    # 3. Export to CSV for Lumerical
    # ---------------------------------------------------------
    export_path = os.path.join(export_dir, 'predicted_target_geometry_polystyrene_inv.csv')

    # Notice the brackets [pred_physical_geom] to keep it in a single row!
    np.savetxt(export_path, [pred_physical_geom], delimiter=",",
               header="Radius1,Theta1,Radius2,Theta2,Radius3,Theta3,Radius4,Theta4",
               comments='')

    print(f"\nSUCCESS! Geometry parameters exported to: {export_path}")

    # ---------------------------------------------------------
    # 4. Print and Plot Results
    # ---------------------------------------------------------
    print("\n--- PREDICTED GEOMETRY ---")
    for i in range(4):
        print(f"Layer {i+1} Radius: {pred_physical_geom[i*2]:.4e} m")
        print(f"Layer {i+1} Theta:  {pred_physical_geom[i*2 + 1]:.4f} rad")

    plt.figure(figsize=(10, 5))
    plt.plot(x_target, target_100_pts, label='Original Target Shape', color='red', linestyle='--')
    plt.plot(x_target, squashed_spectrum, label='Squashed Target (Fed to AI)', color='black', linewidth=2)
    plt.xlabel('Normalized Wavelength', fontsize=12)
    plt.ylabel('Intensity', fontsize=12)
    plt.title('Target Preparation (Path A)', fontsize=14)
    plt.legend()
    plt.grid(True, alpha=0.5)
    plt.show()

    return pred_physical_geom

# ==========================================
# RUN THE PIPELINE:
# ==========================================
csv_path = '/content/drive/MyDrive/PolaritonicONN/TrainingDataset/Target_spectra_polystyrene_inv.csv' # UPDATE THIS
df = pd.read_csv(csv_path)

# Extract Intensity column (assuming it's the last column)
my_target_data = df.iloc[:, -1].values

# Define where you want the new parameter CSV saved
my_export_folder = data_dir # Saves to your Google Drive folder

final_geometry = design_from_target_and_export(my_target_data, spec_min, spec_max, my_export_folder)

# Linear Regression

In [ ]:
import os
import numpy as np
import torch
import joblib
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import train_test_split

# ---------------------------------------------------------
# 1. Device & Setup
# ---------------------------------------------------------
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Make sure data_dir is defined!
# data_dir = '/content/drive/MyDrive/Your_Folder_Path'

# ---------------------------------------------------------
# 2. Load Raw Data & Scalers
# ---------------------------------------------------------
print("--- Loading Data and Scalers ---")
# UPDATE THESE TWO LINES to match your actual raw dataset filenames!
raw_geom = np.load(os.path.join(data_dir, 'geometry_compiled.npy'))
raw_spec = np.load(os.path.join(data_dir, 'spectra_compiled.npy'))

# Load your saved scalers
scaler_geom = joblib.load(os.path.join(data_dir, 'scaler_geom.pkl'))
spec_limits = np.load(os.path.join(data_dir, 'spec_min_max.npy'))
spec_min, spec_max = spec_limits[0], spec_limits[1]

# ---------------------------------------------------------
# 3. Preprocess Data
# ---------------------------------------------------------
# Scale geometry using the trained standard scaler
scaled_geom = scaler_geom.transform(raw_geom)

# Normalize spectra globally using your saved min and max (maps to 0.0 - 1.0)
normalized_spec = (raw_spec - spec_min) / (spec_max - spec_min)

# ---------------------------------------------------------
# 4. Train/Validation Split & DataLoaders
# ---------------------------------------------------------
# Split 80% for training, 20% for validation
X_train, X_val, y_train, y_val = train_test_split(scaled_geom, normalized_spec, test_size=0.2, random_state=42)

# Convert to PyTorch Tensors
X_train_t = torch.tensor(X_train, dtype=torch.float32)
y_train_t = torch.tensor(y_train, dtype=torch.float32)
X_val_t = torch.tensor(X_val, dtype=torch.float32)
y_val_t = torch.tensor(y_val, dtype=torch.float32)

# Create Datasets and Loaders
batch_size = 32 # You can adjust this if needed
train_dataset = TensorDataset(X_train_t, y_train_t)
val_dataset = TensorDataset(X_val_t, y_val_t)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
print(f"DataLoaders built! Training batches: {len(train_loader)} | Val batches: {len(val_loader)}")

# ---------------------------------------------------------
# 5. Define Linear Model & Train
# ---------------------------------------------------------
class ForwardLinear(nn.Module):
    def __init__(self):
        super(ForwardLinear, self).__init__()
        self.linear = nn.Linear(8, 100) # Pure linear mapping, no activation!

    def forward(self, x):
        return self.linear(x)

forward_linear = ForwardLinear().to(device)
criterion = nn.MSELoss()
optimizer_linear = optim.Adam(forward_linear.parameters(), lr=0.001)

epochs = 100
print("\n--- Starting Linear Regression Training ---")

for epoch in range(epochs):
    forward_linear.train()
    running_loss = 0.0

    for geom_batch, spec_batch in train_loader:
        geom_batch, spec_batch = geom_batch.to(device), spec_batch.to(device)

        optimizer_linear.zero_grad()
        predictions = forward_linear(geom_batch)
        loss = criterion(predictions, spec_batch)
        loss.backward()
        optimizer_linear.step()

        running_loss += loss.item()

    avg_train_loss = running_loss / len(train_loader)
    train_losses.append(avg_train_loss)

    forward_linear.eval()
    val_loss = 0.0
    with torch.no_grad():
        for geom_val, spec_val in val_loader:
            geom_val, spec_val = geom_val.to(device), spec_val.to(device)
            val_preds = forward_linear(geom_val)
            v_loss = criterion(val_preds, spec_val)
            val_loss += v_loss.item()

    avg_val_loss = val_loss / len(val_loader)
    val_losses.append(avg_val_loss)

    if (epoch + 1) % 10 == 0:
        print(f"Epoch [{epoch+1}/{epochs}] | Train Loss: {avg_train_loss:.6f} | Val Loss: {avg_val_loss:.6f}")

print("--- Training Complete ---")

import matplotlib.pyplot as plt

def plot_learning_curve(train_losses, val_losses):
    """
    Plots the training and validation loss to visualize where
    the model stops learning (the "floor").
    """
    epochs_range = range(1, len(train_losses) + 1)

    plt.figure(figsize=(8, 5))
    plt.plot(epochs_range, train_losses, label='Training Loss', color='blue', linewidth=2)
    plt.plot(epochs_range, val_losses, label='Validation Loss', color='red', linestyle='--', linewidth=2)

    # Set y-axis to logarithmic scale if you want to see tiny differences better
    # plt.yscale('log')

    plt.xlabel('Epochs', fontsize=12)
    plt.ylabel('Mean Squared Error (Loss)', fontsize=12)
    plt.title('Linear Baseline Learning Curve', fontsize=14)
    plt.legend()
    plt.grid(True, alpha=0.5)
    plt.show()

# ==========================================
# RUN THE PLOT:
# ==========================================
plot_learning_curve(train_losses, val_losses)

# Test different activation functions

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt

# ---------------------------------------------------------
# 1. Define a Dynamic Forward Model
# ---------------------------------------------------------
class DynamicForwardMLP(nn.Module):
    def __init__(self, activation_module):
        super(DynamicForwardMLP, self).__init__()
        # We pass the activation function as a variable!
        self.net = nn.Sequential(
            nn.Linear(8, 64), activation_module(),
            nn.Linear(64, 128), activation_module(),
            nn.Linear(128, 256), activation_module(),
            nn.Linear(256, 100) # Always linear at the end!
        )
    def forward(self, x):
        return self.net(x)

# ---------------------------------------------------------
# 2. Setup the Experiment
# ---------------------------------------------------------
# Dictionary of the functions we want to test
activations_to_test = {
    'ReLU': nn.ReLU,
    'Sigmoid': nn.Sigmoid,
    'Tanh': nn.Tanh
}

# Dictionary to store the loss curves for plotting later
results = {}
epochs = 50 # Keep it to 50 for a quicker comparative test
criterion = nn.MSELoss()

print("--- STARTING ACTIVATION SHOWDOWN ---")

# ---------------------------------------------------------
# 3. Train All Three Models Back-to-Back
# ---------------------------------------------------------
for name, act_module in activations_to_test.items():
    print(f"\nTraining Model with {name} Activation...")

    # Initialize a fresh model and optimizer for this specific activation
    model = DynamicForwardMLP(act_module).to(device)
    optimizer = optim.Adam(model.parameters(), lr=0.001)

    train_losses = []
    val_losses = []

    for epoch in range(epochs):
        # --- Training ---
        model.train()
        running_loss = 0.0
        for spec_batch, geom_batch in train_loader:

            geom_batch, spec_batch = geom_batch.to(device), spec_batch.to(device)

            optimizer.zero_grad()
            preds = model(geom_batch) # Geometry goes in
            loss = criterion(preds, spec_batch) # Spectrum is the target
            loss.backward()
            optimizer.step()
            running_loss += loss.item()

        train_losses.append(running_loss / len(train_loader))

        # --- Validation ---
        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for spec_val, geom_val in val_loader:
                geom_val, spec_val = geom_val.to(device), spec_val.to(device)
                val_preds = model(geom_val)
                val_loss += criterion(val_preds, spec_val).item()

        val_losses.append(val_loss / len(val_loader))

        # Print progress every 10 epochs
        if (epoch + 1) % 10 == 0:
            print(f"   Epoch {epoch+1}/{epochs} | Val Loss: {val_losses[-1]:.6f}")

    # Save the loss histories for plotting
    results[name] = {
        'train': train_losses,
        'val': val_losses
    }

print("\n--- ALL TRAINING COMPLETE. GENERATING PLOT ---")

# ---------------------------------------------------------
# 4. Plot the Ultimate Comparison
# ---------------------------------------------------------
plt.figure(figsize=(10, 6))

# Define specific colors for each activation function
colors = {'ReLU': 'blue', 'Sigmoid': 'green', 'Tanh': 'red'}
epochs_range = range(1, epochs + 1)

for name in results:
    train_line = results[name]['train']
    val_line = results[name]['val']
    c = colors[name]

    # Solid line for Training, Dotted line (':') for Validation
    plt.plot(epochs_range, train_line, color=c, linestyle='-', linewidth=2, label=f'{name} (Train)')
    plt.plot(epochs_range, val_line, color=c, linestyle=':', linewidth=3, label=f'{name} (Validation)')

# Use a logarithmic scale on the Y-axis so we can see the tiny differences at the bottom!
plt.yscale('log')
plt.xlabel('Epochs', fontsize=12)
plt.ylabel('Mean Squared Error (Log Scale)', fontsize=12)
plt.title('Activation Function Performance Comparison', fontsize=14)

# Put the legend outside the plot so it doesn't cover the lines
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.grid(True, alpha=0.5)
plt.tight_layout()
plt.show()

# Check Match and Crosstalk

In [ ]:
import numpy as np
import torch
import matplotlib.pyplot as plt
from scipy.interpolate import interp1d

def prepare_target(raw_target, spec_min, spec_max):
    """Helper function to interpolate and squash a target to 100 points."""
    x_original = np.linspace(0, 1, len(raw_target))
    x_100 = np.linspace(0, 1, 100)

    interpolator = interp1d(x_original, raw_target, kind='cubic')
    target_100 = interpolator(x_100)

    t_min, t_max = np.min(target_100), np.max(target_100)
    squashed = spec_min + ((target_100 - t_min) / (t_max - t_min)) * (spec_max - spec_min)
    return x_100, squashed

def analyze_filter_specificity(raw_target1, raw_target2, spec_min, spec_max, inv_model, fwd_model):
    """
    Predicts a filter for Target 1, then cross-multiplies that predicted
    spectrum against BOTH targets to prove specificity.
    """
    # 1. Prepare both targets
    x_100, target1_squashed = prepare_target(raw_target1, spec_min, spec_max)
    _, target2_squashed = prepare_target(raw_target2, spec_min, spec_max)

    # Normalize Target 1 for the Inverse AI
    t1_norm = (target1_squashed - spec_min) / (spec_max - spec_min)
    t1_tensor = torch.tensor(t1_norm, dtype=torch.float32).unsqueeze(0).to(device)

    # 2. Predict Geometry & Spectrum for Target 1 ONLY
    inv_model.eval()
    fwd_model.eval()
    with torch.no_grad():
        pred_geom1 = inv_model(t1_tensor)
        pred_spec1_norm = fwd_model(pred_geom1)

    pred_spec1 = pred_spec1_norm.squeeze(0).cpu().numpy() * (spec_max - spec_min) + spec_min

    # ---------------------------------------------------------
    # 3. MULTIPLY AND SUM (The Numeric Proof)
    # ---------------------------------------------------------
    # Match: Target 1 * Predicted 1
    match_curve = target1_squashed * pred_spec1
    match_score = np.sum(match_curve)

    # Mismatch (Crosstalk): Target 2 * Predicted 1
    mismatch_curve = target2_squashed * pred_spec1
    mismatch_score = np.sum(mismatch_curve)

    print("\n--- NUMERIC OVERLAP RESULTS ---")
    print(f"Match Score (T1 * P1):    {match_score:.4f}")
    print(f"Mismatch Score (T2 * P1): {mismatch_score:.4f}")
    print(f"Specificity Ratio:        {match_score / mismatch_score:.2f}x better alignment")

    # ---------------------------------------------------------
    # 4. Plot the Comparison
    # ---------------------------------------------------------
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5), sharey=True)

    # Plot 1: The Match
    ax1.plot(x_100, target1_squashed, label='Target pentacene', color='black', linestyle='--')
    ax1.plot(x_100, pred_spec1, label='Predicted pentacene', color='blue')
    ax1.fill_between(x_100, match_curve, color='green', alpha=0.3, label=f'Overlap (Sum: {match_score:.1f})')
    ax1.set_title('The Match (T1 x P1)', fontsize=14)
    ax1.set_xlabel('Normalized Wavelength')
    ax1.set_ylabel('Intensity')
    ax1.legend()
    ax1.grid(True, alpha=0.4)

    # Plot 2: The Mismatch
    ax2.plot(x_100, target2_squashed, label='Target polystyrene', color='black', linestyle='--')
    ax2.plot(x_100, pred_spec1, label='Predicted pentacene', color='blue') # Notice it's the SAME predicted spectrum!
    ax2.fill_between(x_100, mismatch_curve, color='red', alpha=0.3, label=f'Overlap (Sum: {mismatch_score:.1f})')
    ax2.set_title('The Mismatch / Crosstalk (T2 x P1)', fontsize=14)
    ax2.set_xlabel('Normalized Wavelength')
    ax2.legend()
    ax2.grid(True, alpha=0.4)

    plt.tight_layout()
    plt.show()

    return match_score, mismatch_score

# ==========================================
# RUN THE ANALYSIS:
# ==========================================
import numpy as np
import os

# 1. Define the exact paths to your two target files
# (Make sure to update these strings to your actual file names!)
target1_path = os.path.join(data_dir, 'Target_spectra_pentacene.csv')
target2_path = os.path.join(data_dir, 'Target_spectra_polystyrene_cutted-um.csv')

# 2. Load the full CSV data
# Note: If your CSV does NOT have a header row, you can remove 'skiprows=1'
raw_csv1 = np.loadtxt(target1_path, delimiter=',', skiprows=1)
raw_csv2 = np.loadtxt(target2_path, delimiter=',', skiprows=1)

# 3. Slice the arrays to grab ONLY the second column (Intensity)
# In Python, [:, 1] means "give me all the rows (:), but only column index 1"
target1_intensity = raw_csv1[:, 1]
target2_intensity = raw_csv2[:, 1]

# Quick sanity check to prove we isolated the 1D arrays correctly!
print(f"Target 1 loaded shape: {target1_intensity.shape} (Should be something like (36,))")
print(f"Target 2 loaded shape: {target2_intensity.shape}")

# 4. NOW run the analysis using just the pure intensity arrays!
analyze_filter_specificity(target1_intensity, target2_intensity, spec_min, spec_max, inverse_model, forward_model)

In [ ]:
import numpy as np
import torch
import matplotlib.pyplot as plt
from scipy.interpolate import interp1d

def prepare_target(raw_target, spec_min, spec_max):
    """Helper function to interpolate and squash a target to 100 points."""
    x_original = np.linspace(0, 1, len(raw_target))
    x_100 = np.linspace(0, 1, 100)

    interpolator = interp1d(x_original, raw_target, kind='cubic')
    target_100 = interpolator(x_100)

    t_min, t_max = np.min(target_100), np.max(target_100)
    squashed = spec_min + ((target_100 - t_min) / (t_max - t_min)) * (spec_max - spec_min)
    return x_100, squashed

def analyze_filter_specificity(raw_target1, raw_target2, spec_min, spec_max, inv_model, fwd_model):
    """
    Predicts a filter for Target 1, then cross-multiplies that predicted
    spectrum against BOTH targets to prove specificity.
    """
    # 1. Prepare both targets
    x_100, target1_squashed = prepare_target(raw_target1, spec_min, spec_max)
    _, target2_squashed = prepare_target(raw_target2, spec_min, spec_max)

    # Normalize Target 1 for the Inverse AI
    t1_norm = (target1_squashed - spec_min) / (spec_max - spec_min)
    t1_tensor = torch.tensor(t1_norm, dtype=torch.float32).unsqueeze(0).to(device)

    # 2. Predict Geometry & Spectrum for Target 1 ONLY
    inv_model.eval()
    fwd_model.eval()
    with torch.no_grad():
        pred_geom1 = inv_model(t1_tensor)
        pred_spec1_norm = fwd_model(pred_geom1)

    pred_spec1 = pred_spec1_norm.squeeze(0).cpu().numpy() * (spec_max - spec_min) + spec_min

    # ---------------------------------------------------------
    # 3. MULTIPLY AND SUM (The Numeric Proof)
    # ---------------------------------------------------------
    # Match: Target 1 * Predicted 1
    match_curve = target1_squashed * pred_spec1
    match_score = np.sum(match_curve)

    # Mismatch (Crosstalk): Target 2 * Predicted 1
    mismatch_curve = target2_squashed * pred_spec1
    mismatch_score = np.sum(mismatch_curve)

    print("\n--- NUMERIC OVERLAP RESULTS ---")
    print(f"Match Score (T1 * P1):    {match_score:.4f}")
    print(f"Mismatch Score (T2 * P1): {mismatch_score:.4f}")
    print(f"Specificity Ratio:        {match_score / mismatch_score:.2f}x better alignment")

    # ---------------------------------------------------------
    # 4. Plot the Comparison
    # ---------------------------------------------------------
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5), sharey=True)

    # Plot 1: The Match
    ax1.plot(x_100, target1_squashed, label='Target polystyrene', color='black', linestyle='--')
    ax1.plot(x_100, pred_spec1, label='Predicted polystyrene', color='blue')
    ax1.fill_between(x_100, match_curve, color='green', alpha=0.3, label=f'Overlap (Sum: {match_score:.1f})')
    ax1.set_title('The Match (T1 x P1)', fontsize=14)
    ax1.set_xlabel('Normalized Wavelength')
    ax1.set_ylabel('Intensity')
    ax1.legend()
    ax1.grid(True, alpha=0.4)

    # Plot 2: The Mismatch
    ax2.plot(x_100, target2_squashed, label='Target pentacene', color='black', linestyle='--')
    ax2.plot(x_100, pred_spec1, label='Predicted polystyrene', color='blue') # Notice it's the SAME predicted spectrum!
    ax2.fill_between(x_100, mismatch_curve, color='red', alpha=0.3, label=f'Overlap (Sum: {mismatch_score:.1f})')
    ax2.set_title('The Mismatch / Crosstalk (T2 x P1)', fontsize=14)
    ax2.set_xlabel('Normalized Wavelength')
    ax2.legend()
    ax2.grid(True, alpha=0.4)

    plt.tight_layout()
    plt.show()

    return match_score, mismatch_score

# ==========================================
# RUN THE ANALYSIS:
# ==========================================
import numpy as np
import os

# 1. Define the exact paths to your two target files
# (Make sure to update these strings to your actual file names!)
target1_path = os.path.join(data_dir, 'Target_spectra_polystyrene_cutted-um.csv')
target2_path = os.path.join(data_dir, 'Target_spectra_pentacene.csv')

# 2. Load the full CSV data
# Note: If your CSV does NOT have a header row, you can remove 'skiprows=1'
raw_csv1 = np.loadtxt(target1_path, delimiter=',', skiprows=1)
raw_csv2 = np.loadtxt(target2_path, delimiter=',', skiprows=1)

# 3. Slice the arrays to grab ONLY the second column (Intensity)
# In Python, [:, 1] means "give me all the rows (:), but only column index 1"
target1_intensity = raw_csv1[:, 1]
target2_intensity = raw_csv2[:, 1]

# Quick sanity check to prove we isolated the 1D arrays correctly!
print(f"Target 1 loaded shape: {target1_intensity.shape} (Should be something like (36,))")
print(f"Target 2 loaded shape: {target2_intensity.shape}")

# 4. NOW run the analysis using just the pure intensity arrays!
analyze_filter_specificity(target1_intensity, target2_intensity, spec_min, spec_max, inverse_model, forward_model)

In [ ]:
import numpy as np
import torch
import matplotlib.pyplot as plt
from scipy.interpolate import interp1d

def prepare_target(raw_target, spec_min, spec_max):
    """Helper function to interpolate and squash a target to 100 points."""
    x_original = np.linspace(0, 1, len(raw_target))
    x_100 = np.linspace(0, 1, 100)

    interpolator = interp1d(x_original, raw_target, kind='cubic')
    target_100 = interpolator(x_100)

    t_min, t_max = np.min(target_100), np.max(target_100)
    squashed = spec_min + ((target_100 - t_min) / (t_max - t_min)) * (spec_max - spec_min)
    return x_100, squashed

def analyze_filter_specificity(raw_target1, raw_target2, spec_min, spec_max, inv_model, fwd_model):
    """
    Predicts a filter for Target 1, then cross-multiplies that predicted
    spectrum against BOTH targets to prove specificity.
    """
    # 1. Prepare both targets
    x_100, target1_squashed = prepare_target(raw_target1, spec_min, spec_max)
    _, target2_squashed = prepare_target(raw_target2, spec_min, spec_max)

    # Normalize Target 1 for the Inverse AI
    t1_norm = (target1_squashed - spec_min) / (spec_max - spec_min)
    t1_tensor = torch.tensor(t1_norm, dtype=torch.float32).unsqueeze(0).to(device)

    # 2. Predict Geometry & Spectrum for Target 1 ONLY
    inv_model.eval()
    fwd_model.eval()
    with torch.no_grad():
        pred_geom1 = inv_model(t1_tensor)
        pred_spec1_norm = fwd_model(pred_geom1)

    pred_spec1 = pred_spec1_norm.squeeze(0).cpu().numpy() * (spec_max - spec_min) + spec_min

    # ---------------------------------------------------------
    # 3. MULTIPLY AND SUM (The Numeric Proof)
    # ---------------------------------------------------------
    # Match: Target 1 * Predicted 1
    match_curve = target1_squashed * pred_spec1
    match_score = np.sum(match_curve)

    # Mismatch (Crosstalk): Target 2 * Predicted 1
    mismatch_curve = target2_squashed * pred_spec1
    mismatch_score = np.sum(mismatch_curve)

    print("\n--- NUMERIC OVERLAP RESULTS ---")
    print(f"Match Score (T1 * P1):    {match_score:.4f}")
    print(f"Mismatch Score (T2 * P1): {mismatch_score:.4f}")
    print(f"Specificity Ratio:        {match_score / mismatch_score:.2f}x better alignment")

    # ---------------------------------------------------------
    # 4. Plot the Comparison
    # ---------------------------------------------------------
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5), sharey=True)

    # Plot 1: The Match
    ax1.plot(x_100, target1_squashed, label='Target pentacene', color='black', linestyle='--')
    ax1.plot(x_100, pred_spec1, label='Predicted pentacene', color='blue')
    ax1.fill_between(x_100, match_curve, color='green', alpha=0.3, label=f'Overlap (Sum: {match_score:.1f})')
    ax1.set_title('The Match (T1 x P1)', fontsize=14)
    ax1.set_xlabel('Normalized Wavelength')
    ax1.set_ylabel('Intensity')
    ax1.legend()
    ax1.grid(True, alpha=0.4)

    # Plot 2: The Mismatch
    ax2.plot(x_100, target2_squashed, label='Target polystyrene', color='black', linestyle='--')
    ax2.plot(x_100, pred_spec1, label='Predicted pentacene', color='blue') # Notice it's the SAME predicted spectrum!
    ax2.fill_between(x_100, mismatch_curve, color='red', alpha=0.3, label=f'Overlap (Sum: {mismatch_score:.1f})')
    ax2.set_title('The Mismatch / Crosstalk (T2 x P1)', fontsize=14)
    ax2.set_xlabel('Normalized Wavelength')
    ax2.legend()
    ax2.grid(True, alpha=0.4)

    plt.tight_layout()
    plt.show()

    return match_score, mismatch_score

# ==========================================
# RUN THE ANALYSIS:
# ==========================================
import numpy as np
import os

# 1. Define the exact paths to your two target files
# (Make sure to update these strings to your actual file names!)
target1_path = os.path.join(data_dir, 'Target_spectra_pentacene_inv.csv')
target2_path = os.path.join(data_dir, 'Target_spectra_polystyrene_inv.csv')

# 2. Load the full CSV data
# Note: If your CSV does NOT have a header row, you can remove 'skiprows=1'
raw_csv1 = np.loadtxt(target1_path, delimiter=',', skiprows=1)
raw_csv2 = np.loadtxt(target2_path, delimiter=',', skiprows=1)

# 3. Slice the arrays to grab ONLY the second column (Intensity)
# In Python, [:, 1] means "give me all the rows (:), but only column index 1"
target1_intensity = raw_csv1[:, 1]
target2_intensity = raw_csv2[:, 1]

# Quick sanity check to prove we isolated the 1D arrays correctly!
print(f"Target 1 loaded shape: {target1_intensity.shape} (Should be something like (36,))")
print(f"Target 2 loaded shape: {target2_intensity.shape}")

# 4. NOW run the analysis using just the pure intensity arrays!
analyze_filter_specificity(target1_intensity, target2_intensity, spec_min, spec_max, inverse_model, forward_model)

In [ ]:
import numpy as np
import torch
import matplotlib.pyplot as plt
from scipy.interpolate import interp1d

def prepare_target(raw_target, spec_min, spec_max):
    """Helper function to interpolate and squash a target to 100 points."""
    x_original = np.linspace(0, 1, len(raw_target))
    x_100 = np.linspace(0, 1, 100)

    interpolator = interp1d(x_original, raw_target, kind='cubic')
    target_100 = interpolator(x_100)

    t_min, t_max = np.min(target_100), np.max(target_100)
    squashed = spec_min + ((target_100 - t_min) / (t_max - t_min)) * (spec_max - spec_min)
    return x_100, squashed

def analyze_filter_specificity(raw_target1, raw_target2, spec_min, spec_max, inv_model, fwd_model):
    """
    Predicts a filter for Target 1, then cross-multiplies that predicted
    spectrum against BOTH targets to prove specificity.
    """
    # 1. Prepare both targets
    x_100, target1_squashed = prepare_target(raw_target1, spec_min, spec_max)
    _, target2_squashed = prepare_target(raw_target2, spec_min, spec_max)

    # Normalize Target 1 for the Inverse AI
    t1_norm = (target1_squashed - spec_min) / (spec_max - spec_min)
    t1_tensor = torch.tensor(t1_norm, dtype=torch.float32).unsqueeze(0).to(device)

    # 2. Predict Geometry & Spectrum for Target 1 ONLY
    inv_model.eval()
    fwd_model.eval()
    with torch.no_grad():
        pred_geom1 = inv_model(t1_tensor)
        pred_spec1_norm = fwd_model(pred_geom1)

    pred_spec1 = pred_spec1_norm.squeeze(0).cpu().numpy() * (spec_max - spec_min) + spec_min

    # ---------------------------------------------------------
    # 3. MULTIPLY AND SUM (The Numeric Proof)
    # ---------------------------------------------------------
    # Match: Target 1 * Predicted 1
    match_curve = target1_squashed * pred_spec1
    match_score = np.sum(match_curve)

    # Mismatch (Crosstalk): Target 2 * Predicted 1
    mismatch_curve = target2_squashed * pred_spec1
    mismatch_score = np.sum(mismatch_curve)

    print("\n--- NUMERIC OVERLAP RESULTS ---")
    print(f"Match Score (T1 * P1):    {match_score:.4f}")
    print(f"Mismatch Score (T2 * P1): {mismatch_score:.4f}")
    print(f"Specificity Ratio:        {match_score / mismatch_score:.2f}x better alignment")

    # ---------------------------------------------------------
    # 4. Plot the Comparison
    # ---------------------------------------------------------
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5), sharey=True)

    # Plot 1: The Match
    ax1.plot(x_100, target1_squashed, label='Target polystyrene', color='black', linestyle='--')
    ax1.plot(x_100, pred_spec1, label='Predicted polystyrene', color='blue')
    ax1.fill_between(x_100, match_curve, color='green', alpha=0.3, label=f'Overlap (Sum: {match_score:.1f})')
    ax1.set_title('The Match (T1 x P1)', fontsize=14)
    ax1.set_xlabel('Normalized Wavelength')
    ax1.set_ylabel('Intensity')
    ax1.legend()
    ax1.grid(True, alpha=0.4)

    # Plot 2: The Mismatch
    ax2.plot(x_100, target2_squashed, label='Target pentacene', color='black', linestyle='--')
    ax2.plot(x_100, pred_spec1, label='Predicted polystyrene', color='blue') # Notice it's the SAME predicted spectrum!
    ax2.fill_between(x_100, mismatch_curve, color='red', alpha=0.3, label=f'Overlap (Sum: {mismatch_score:.1f})')
    ax2.set_title('The Mismatch / Crosstalk (T2 x P1)', fontsize=14)
    ax2.set_xlabel('Normalized Wavelength')
    ax2.legend()
    ax2.grid(True, alpha=0.4)

    plt.tight_layout()
    plt.show()

    return match_score, mismatch_score

# ==========================================
# RUN THE ANALYSIS:
# ==========================================
import numpy as np
import os

# 1. Define the exact paths to your two target files
# (Make sure to update these strings to your actual file names!)
target1_path = os.path.join(data_dir, 'Target_spectra_polystyrene_inv.csv')
target2_path = os.path.join(data_dir, 'Target_spectra_pentacene_inv.csv')

# 2. Load the full CSV data
# Note: If your CSV does NOT have a header row, you can remove 'skiprows=1'
raw_csv1 = np.loadtxt(target1_path, delimiter=',', skiprows=1)
raw_csv2 = np.loadtxt(target2_path, delimiter=',', skiprows=1)

# 3. Slice the arrays to grab ONLY the second column (Intensity)
# In Python, [:, 1] means "give me all the rows (:), but only column index 1"
target1_intensity = raw_csv1[:, 1]
target2_intensity = raw_csv2[:, 1]

# Quick sanity check to prove we isolated the 1D arrays correctly!
print(f"Target 1 loaded shape: {target1_intensity.shape} (Should be something like (36,))")
print(f"Target 2 loaded shape: {target2_intensity.shape}")

# 4. NOW run the analysis using just the pure intensity arrays!
analyze_filter_specificity(target1_intensity, target2_intensity, spec_min, spec_max, inverse_model, forward_model)

In [ ]:
import numpy as np
import torch
import matplotlib.pyplot as plt
from scipy.interpolate import interp1d

def prepare_target(raw_target, spec_min, spec_max):
    """Interpolates target to 100 points and maps to physical bounds."""
    x_original = np.linspace(0, 1, len(raw_target))
    x_100 = np.linspace(0, 1, 100)
    interpolator = interp1d(x_original, raw_target, kind='cubic')
    target_100 = interpolator(x_100)

    t_min, t_max = np.min(target_100), np.max(target_100)
    squashed = spec_min + ((target_100 - t_min) / (t_max - t_min)) * (spec_max - spec_min)
    return x_100, squashed

def analyze_2x2_crosstalk(t1_raw, t2_raw, spec_min, spec_max, inv_model, fwd_model, name1="Target 1", name2="Target 2"):
    # ---------------------------------------------------------
    # 1. Prepare Targets and Tensors
    # ---------------------------------------------------------
    x_100, t1_phys = prepare_target(t1_raw, spec_min, spec_max)
    _, t2_phys = prepare_target(t2_raw, spec_min, spec_max)

    t1_tensor = torch.tensor((t1_phys - spec_min) / (spec_max - spec_min), dtype=torch.float32).unsqueeze(0).to(device)
    t2_tensor = torch.tensor((t2_phys - spec_min) / (spec_max - spec_min), dtype=torch.float32).unsqueeze(0).to(device)

    # ---------------------------------------------------------
    # 2. Predict Spectrums for BOTH Targets
    # ---------------------------------------------------------
    inv_model.eval()
    fwd_model.eval()
    with torch.no_grad():
        pred_spec1 = fwd_model(inv_model(t1_tensor)).squeeze(0).cpu().numpy() * (spec_max - spec_min) + spec_min
        pred_spec2 = fwd_model(inv_model(t2_tensor)).squeeze(0).cpu().numpy() * (spec_max - spec_min) + spec_min

    # ---------------------------------------------------------
    # 3. Calculate the 4 Overlaps
    # ---------------------------------------------------------
    match_11 = t1_phys * pred_spec1
    miss_21  = t2_phys * pred_spec1
    miss_12  = t1_phys * pred_spec2
    match_22 = t2_phys * pred_spec2

    sum_11, sum_21 = np.sum(match_11), np.sum(miss_21)
    sum_12, sum_22 = np.sum(miss_12), np.sum(match_22)

    # ---------------------------------------------------------
    # 4. Plot the 2x2 Grid
    # ---------------------------------------------------------
    fig, ax = plt.subplots(2, 2, figsize=(14, 10), sharex=True, sharey=True)

    # --- ROW 1: Testing Filter 1 ---
    # Top-Left: T1 x P1 (Match)
    ax[0, 0].plot(x_100, t1_phys, 'k--', label=name1)
    ax[0, 0].plot(x_100, pred_spec1, 'b-', label='Predicted pentacene')
    ax[0, 0].fill_between(x_100, match_11, color='green', alpha=0.3, label=f'Overlap: {sum_11:.1f}')
    ax[0, 0].set_title(f'MATCH: {name1} x Predicted pentacene', fontsize=12)
    ax[0, 0].legend()
    ax[0, 0].grid(True, alpha=0.4)

    # Top-Right: T2 x P1 (Mismatch)
    ax[0, 1].plot(x_100, t2_phys, 'k--', label=name2)
    ax[0, 1].plot(x_100, pred_spec1, 'b-', label='Predicted pentacene')
    ax[0, 1].fill_between(x_100, miss_21, color='red', alpha=0.3, label=f'Overlap: {sum_21:.1f}')
    ax[0, 1].set_title(f'CROSSTALK: {name2} x Predicted pentacene', fontsize=12)
    ax[0, 1].legend()
    ax[0, 1].grid(True, alpha=0.4)

    # --- ROW 2: Testing Filter 2 ---
    # Bottom-Left: T1 x P2 (Mismatch)
    ax[1, 0].plot(x_100, t1_phys, 'k--', label=name1)
    ax[1, 0].plot(x_100, pred_spec2, 'm-', label='Predicted polystyrene') # Magenta for Filter 2
    ax[1, 0].fill_between(x_100, miss_12, color='red', alpha=0.3, label=f'Overlap: {sum_12:.1f}')
    ax[1, 0].set_title(f'CROSSTALK: {name1} x Predicted polystyrene', fontsize=12)
    ax[1, 0].set_xlabel('Normalized Wavelength')
    ax[1, 0].legend()
    ax[1, 0].grid(True, alpha=0.4)

    # Bottom-Right: T2 x P2 (Match)
    ax[1, 1].plot(x_100, t2_phys, 'k--', label=name2)
    ax[1, 1].plot(x_100, pred_spec2, 'm-', label='Predicted polystyrene')
    ax[1, 1].fill_between(x_100, match_22, color='green', alpha=0.3, label=f'Overlap: {sum_22:.1f}')
    ax[1, 1].set_title(f'MATCH: {name2} x Predicted polystyrene', fontsize=12)
    ax[1, 1].set_xlabel('Normalized Wavelength')
    ax[1, 1].legend()
    ax[1, 1].grid(True, alpha=0.4)

    plt.tight_layout()
    plt.show()

# 1. Define the exact paths to your two target files
# (Make sure to update these strings to your actual file names!)
target1_path = os.path.join(data_dir, 'Target_spectra_pentacene.csv')
target2_path = os.path.join(data_dir, 'Target_spectra_polystyrene_cutted-um.csv')

# 2. Load the full CSV data
# Note: If your CSV does NOT have a header row, you can remove 'skiprows=1'
raw_csv1 = np.loadtxt(target1_path, delimiter=',', skiprows=1)
raw_csv2 = np.loadtxt(target2_path, delimiter=',', skiprows=1)

# 3. Slice the arrays to grab ONLY the second column (Intensity)
# In Python, [:, 1] means "give me all the rows (:), but only column index 1"
target1_intensity = raw_csv1[:, 1]
target2_intensity = raw_csv2[:, 1]

# Quick sanity check to prove we isolated the 1D arrays correctly!
print(f"Target 1 loaded shape: {target1_intensity.shape} (Should be something like (36,))")
print(f"Target 2 loaded shape: {target2_intensity.shape}")


# Run it like this:
analyze_2x2_crosstalk(target1_intensity, target2_intensity, spec_min, spec_max, inverse_model, forward_model, "Pentacene", "Polystyrene")

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.interpolate import interp1d

# ==========================================
# 1. BULLETPROOF DATA LOADER
# ==========================================
def load_and_force_math(filepath):
    """
    Reads a CSV, strips away ALL text/quotes/tags, forces the pure numbers into floats,
    and instantly deletes any blank rows.
    """
    # 1. Load blindly as raw text (dtype=str) so we can scrub it safely
    df = pd.read_csv(filepath, header=None, usecols=[0, 1], dtype=str)

    # 2. THE TITANIUM SCRUB: Delete ANY character that isn't a number (0-9), a decimal (.), or a minus (-)
    df[0] = df[0].str.replace(r'[^\d.-]', '', regex=True)
    df[1] = df[1].str.replace(r'[^\d.-]', '', regex=True)

    # 3. Force conversion to math floats. Any completely empty cells become NaN
    df[0] = pd.to_numeric(df[0], errors='coerce')
    df[1] = pd.to_numeric(df[1], errors='coerce')

    # 4. Delete the NaNs
    df = df.dropna()

    # 5. Safety check so we don't get a blind error!
    if len(df) == 0:
        print(f"🚨 WARNING: {filepath} is completely empty after cleaning. Please check the raw file formatting!")

    return df.to_numpy()

# ==========================================
# 2. SAFE INTERPOLATION FUNCTION
# ==========================================
def align_to_shared_grid(x_raw, y_raw, shared_x):
    """
    Resamples spectrum. 'fill_value="extrapolate"' prevents crashes
    if the grid is a tiny microscopic fraction of a nanometer longer than the data.
    """
    interpolator = interp1d(x_raw, y_raw, kind='cubic', bounds_error=False, fill_value="extrapolate")
    return interpolator(shared_x)

# ==========================================
# 3. MAIN PLOTTING FUNCTION
# ==========================================

def plot_simulated_crosstalk(t1_data, t2_data, sim1_data, sim2_data, name1="Target 1", name2="Target 2"):
    # 1. Unpack Wavelengths (X) and Intensities (Y)
    t1_x, t1_y = t1_data[:, 0], t1_data[:, 1]
    t2_x, t2_y = t2_data[:, 0], t2_data[:, 1]
    sim1_x, sim1_y = sim1_data[:, 0], sim1_data[:, 1]
    sim2_x, sim2_y = sim2_data[:, 0], sim2_data[:, 1]

    # ==========================================
    # MIN-MAX NORMALIZATION
    # ==========================================
    # This forces every single curve to have a minimum of 0 and a peak of 1
    # so we can fairly compare their physical shapes and overlaps!

    peak_height = 1.0 # Change this to 0.3 if you want them to stop at 0.3!

    t1_y = ((t1_y - t1_y.min()) / (t1_y.max() - t1_y.min())) * peak_height
    t2_y = ((t2_y - t2_y.min()) / (t2_y.max() - t2_y.min())) * peak_height

    sim1_y = ((sim1_y - sim1_y.min()) / (sim1_y.max() - sim1_y.min())) * peak_height
    sim2_y = ((sim2_y - sim2_y.min()) / (sim2_y.max() - sim2_y.min())) * peak_height
    # ==========================================

    # Create the "Safe Zone" Shared Grid
    # We find the highest start point and the lowest end point across all 4 files
    safe_min = max(t1_x.min(), t2_x.min(), sim1_x.min(), sim2_x.min())
    safe_max = min(t1_x.max(), t2_x.max(), sim1_x.max(), sim2_x.max())
    shared_x = np.linspace(safe_min, safe_max, 200)

    # Synchronize all four curves onto this exact same 200-point grid
    t1_aligned = align_to_shared_grid(t1_x, t1_y, shared_x)
    t2_aligned = align_to_shared_grid(t2_x, t2_y, shared_x)
    sim1_aligned = align_to_shared_grid(sim1_x, sim1_y, shared_x)
    sim2_aligned = align_to_shared_grid(sim2_x, sim2_y, shared_x)

    # Calculate Overlaps
    match_11 = t1_aligned * sim1_aligned
    miss_21  = t2_aligned * sim1_aligned
    miss_12  = t1_aligned * sim2_aligned
    match_22 = t2_aligned * sim2_aligned

    sum_11, sum_21 = np.sum(match_11), np.sum(miss_21)
    sum_12, sum_22 = np.sum(miss_12), np.sum(match_22)

    # Plot the 2x2 Grid
    fig, ax = plt.subplots(2, 2, figsize=(14, 10), sharex=True, sharey=True)

    # --- ROW 1: Sim Filter 1 ---
    ax[0, 0].plot(shared_x, t1_aligned, 'k--', label=name1)
    ax[0, 0].plot(shared_x, sim1_aligned, 'b-', label='Predicted pentacene')
    ax[0, 0].fill_between(shared_x, match_11, color='green', alpha=0.3, label=f'Overlap: {sum_11:.1f}')
    ax[0, 0].set_title(f'MATCH: {name1} x Predicted pentacene', fontsize=12)
    ax[0, 0].legend()
    ax[0, 0].grid(True, alpha=0.4)

    ax[0, 1].plot(shared_x, t2_aligned, 'k--', label=name2)
    ax[0, 1].plot(shared_x, sim1_aligned, 'b-', label='Predicted pentacene')
    ax[0, 1].fill_between(shared_x, miss_21, color='red', alpha=0.3, label=f'Overlap: {sum_21:.1f}')
    ax[0, 1].set_title(f'CROSSTALK: {name2} x Predicted pentacene', fontsize=12)
    ax[0, 1].legend()
    ax[0, 1].grid(True, alpha=0.4)

    # --- ROW 2: Sim Filter 2 ---
    ax[1, 0].plot(shared_x, t1_aligned, 'k--', label=name1)
    ax[1, 0].plot(shared_x, sim2_aligned, 'm-', label='Predicted polystyrene')
    ax[1, 0].fill_between(shared_x, miss_12, color='red', alpha=0.3, label=f'Overlap: {sum_12:.1f}')
    ax[1, 0].set_title(f'CROSSTALK: {name1} x Predicted polystyrene', fontsize=12)
    ax[1, 0].set_xlabel('Wavelength (Physical Units)')
    ax[1, 0].legend()
    ax[1, 0].grid(True, alpha=0.4)

    ax[1, 1].plot(shared_x, t2_aligned, 'k--', label=name2)
    ax[1, 1].plot(shared_x, sim2_aligned, 'm-', label='Predicted polystyrene')
    ax[1, 1].fill_between(shared_x, match_22, color='green', alpha=0.3, label=f'Overlap: {sum_22:.1f}')
    ax[1, 1].set_title(f'MATCH: {name2} x Predicted polystyrene', fontsize=12)
    ax[1, 1].set_xlabel('Wavelength (Physical Units)')
    ax[1, 1].legend()
    ax[1, 1].grid(True, alpha=0.4)

    plt.tight_layout()
    plt.show()

# ==========================================
# 4. EXECUTE SCRIPT
# ==========================================

# NOTE: Paste your exact folder path below, making sure it ends with a slash (/)!
data_dir = "/content/drive/MyDrive/PolaritonicONN/TrainingDataset/" # Replace this!

t1_csv = load_and_force_math(f"{data_dir}Target_spectra_pentacene_inv.csv")
t2_csv = load_and_force_math(f"{data_dir}Target_spectra_polystyrene_inv.csv")

sim1_csv = load_and_force_math(f"{data_dir}Pentacene_inv_spectrum.csv")
sim2_csv = load_and_force_math(f"{data_dir}Polystyrene_inv_spectrum.csv")

# Generate the plot!
plot_simulated_crosstalk(t1_csv, t2_csv, sim1_csv, sim2_csv, "Pentacene", "Polystyrene")

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.interpolate import interp1d

# ==========================================
# 1. BULLETPROOF DATA LOADER
# ==========================================
def load_and_force_math(filepath):
    """
    Reads a CSV, strips away ALL text/quotes/tags, forces the pure numbers into floats,
    and instantly deletes any blank rows.
    """
    # 1. Load blindly as raw text (dtype=str) so we can scrub it safely
    df = pd.read_csv(filepath, header=None, usecols=[0, 1], dtype=str)

    # 2. THE TITANIUM SCRUB: Delete ANY character that isn't a number (0-9), a decimal (.), or a minus (-)
    df[0] = df[0].str.replace(r'[^\d.-]', '', regex=True)
    df[1] = df[1].str.replace(r'[^\d.-]', '', regex=True)

    # 3. Force conversion to math floats. Any completely empty cells become NaN
    df[0] = pd.to_numeric(df[0], errors='coerce')
    df[1] = pd.to_numeric(df[1], errors='coerce')

    # 4. Delete the NaNs
    df = df.dropna()

    # 5. Safety check so we don't get a blind error!
    if len(df) == 0:
        print(f"🚨 WARNING: {filepath} is completely empty after cleaning. Please check the raw file formatting!")

    return df.to_numpy()

# ==========================================
# 2. SAFE INTERPOLATION FUNCTION
# ==========================================
def align_to_shared_grid(x_raw, y_raw, shared_x):
    """
    Resamples spectrum. 'fill_value="extrapolate"' prevents crashes
    if the grid is a tiny microscopic fraction of a nanometer longer than the data.
    """
    interpolator = interp1d(x_raw, y_raw, kind='cubic', bounds_error=False, fill_value="extrapolate")
    return interpolator(shared_x)

# ==========================================
# 3. MAIN PLOTTING FUNCTION
# ==========================================

def plot_simulated_crosstalk(t1_data, t2_data, sim1_data, sim2_data, name1="Target 1", name2="Target 2"):
    # 1. Unpack Wavelengths (X) and Intensities (Y)
    t1_x, t1_y = t1_data[:, 0], t1_data[:, 1]
    t2_x, t2_y = t2_data[:, 0], t2_data[:, 1]
    sim1_x, sim1_y = sim1_data[:, 0], sim1_data[:, 1]
    sim2_x, sim2_y = sim2_data[:, 0], sim2_data[:, 1]

    # ==========================================
    # MIN-MAX NORMALIZATION
    # ==========================================
    # This forces every single curve to have a minimum of 0 and a peak of 1
    # so we can fairly compare their physical shapes and overlaps!

    peak_height = 1.0 # Change this to 0.3 if you want them to stop at 0.3!

    t1_y = ((t1_y - t1_y.min()) / (t1_y.max() - t1_y.min())) * peak_height
    t2_y = ((t2_y - t2_y.min()) / (t2_y.max() - t2_y.min())) * peak_height

    sim1_y = ((sim1_y - sim1_y.min()) / (sim1_y.max() - sim1_y.min())) * peak_height
    sim2_y = ((sim2_y - sim2_y.min()) / (sim2_y.max() - sim2_y.min())) * peak_height
    # ==========================================

    # Create the "Safe Zone" Shared Grid
    # We find the highest start point and the lowest end point across all 4 files
    safe_min = max(t1_x.min(), t2_x.min(), sim1_x.min(), sim2_x.min())
    safe_max = min(t1_x.max(), t2_x.max(), sim1_x.max(), sim2_x.max())
    shared_x = np.linspace(safe_min, safe_max, 200)

    # Synchronize all four curves onto this exact same 200-point grid
    t1_aligned = align_to_shared_grid(t1_x, t1_y, shared_x)
    t2_aligned = align_to_shared_grid(t2_x, t2_y, shared_x)
    sim1_aligned = align_to_shared_grid(sim1_x, sim1_y, shared_x)
    sim2_aligned = align_to_shared_grid(sim2_x, sim2_y, shared_x)

    # Calculate Overlaps
    match_11 = t1_aligned * sim1_aligned
    miss_21  = t2_aligned * sim1_aligned
    miss_12  = t1_aligned * sim2_aligned
    match_22 = t2_aligned * sim2_aligned

    sum_11, sum_21 = np.sum(match_11), np.sum(miss_21)
    sum_12, sum_22 = np.sum(miss_12), np.sum(match_22)

    # Plot the 2x2 Grid
    fig, ax = plt.subplots(2, 2, figsize=(14, 10), sharex=True, sharey=True)

    # --- ROW 1: Sim Filter 1 ---
    ax[0, 0].plot(shared_x, t1_aligned, 'k--', label=name1)
    ax[0, 0].plot(shared_x, sim1_aligned, 'b-', label='Predicted pentacene')
    ax[0, 0].fill_between(shared_x, match_11, color='green', alpha=0.3, label=f'Overlap: {sum_11:.1f}')
    ax[0, 0].set_title(f'MATCH: {name1} x Predicted pentacene', fontsize=12)
    ax[0, 0].legend()
    ax[0, 0].grid(True, alpha=0.4)

    ax[0, 1].plot(shared_x, t2_aligned, 'k--', label=name2)
    ax[0, 1].plot(shared_x, sim1_aligned, 'b-', label='Predicted pentacene')
    ax[0, 1].fill_between(shared_x, miss_21, color='red', alpha=0.3, label=f'Overlap: {sum_21:.1f}')
    ax[0, 1].set_title(f'CROSSTALK: {name2} x Predicted pentacene', fontsize=12)
    ax[0, 1].legend()
    ax[0, 1].grid(True, alpha=0.4)

    # --- ROW 2: Sim Filter 2 ---
    ax[1, 0].plot(shared_x, t1_aligned, 'k--', label=name1)
    ax[1, 0].plot(shared_x, sim2_aligned, 'm-', label='Predicted polystyrene')
    ax[1, 0].fill_between(shared_x, miss_12, color='red', alpha=0.3, label=f'Overlap: {sum_12:.1f}')
    ax[1, 0].set_title(f'CROSSTALK: {name1} x Predicted polystyrene', fontsize=12)
    ax[1, 0].set_xlabel('Wavelength (Physical Units)')
    ax[1, 0].legend()
    ax[1, 0].grid(True, alpha=0.4)

    ax[1, 1].plot(shared_x, t2_aligned, 'k--', label=name2)
    ax[1, 1].plot(shared_x, sim2_aligned, 'm-', label='Predicted polystyrene')
    ax[1, 1].fill_between(shared_x, match_22, color='green', alpha=0.3, label=f'Overlap: {sum_22:.1f}')
    ax[1, 1].set_title(f'MATCH: {name2} x Predicted polystyrene', fontsize=12)
    ax[1, 1].set_xlabel('Wavelength (Physical Units)')
    ax[1, 1].legend()
    ax[1, 1].grid(True, alpha=0.4)

    plt.tight_layout()
    plt.show()

# ==========================================
# 4. EXECUTE SCRIPT
# ==========================================

# NOTE: Paste your exact folder path below, making sure it ends with a slash (/)!
data_dir = "/content/drive/MyDrive/PolaritonicONN/TrainingDataset/" # Replace this!

t1_csv = load_and_force_math(f"{data_dir}Target_spectra_pentacene_inv.csv")
t2_csv = load_and_force_math(f"{data_dir}Target_spectra_polystyrene_inv.csv")

sim1_csv = load_and_force_math(f"{data_dir}Pentacene_inv_spectrum+.csv")
sim2_csv = load_and_force_math(f"{data_dir}Polystyrene_inv_spectrum+.csv")

# Generate the plot!
plot_simulated_crosstalk(t1_csv, t2_csv, sim1_csv, sim2_csv, "Pentacene", "Polystyrene")

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.interpolate import interp1d

# ==========================================
# 1. BULLETPROOF DATA LOADER
# ==========================================
def load_and_force_math(filepath):
    """
    Reads a CSV, strips away ALL text/quotes/tags, forces the pure numbers into floats,
    and instantly deletes any blank rows.
    """
    # 1. Load blindly as raw text (dtype=str) so we can scrub it safely
    df = pd.read_csv(filepath, header=None, usecols=[0, 1], dtype=str)

    # 2. THE TITANIUM SCRUB: Delete ANY character that isn't a number (0-9), a decimal (.), or a minus (-)
    df[0] = df[0].str.replace(r'[^\d.-]', '', regex=True)
    df[1] = df[1].str.replace(r'[^\d.-]', '', regex=True)

    # 3. Force conversion to math floats. Any completely empty cells become NaN
    df[0] = pd.to_numeric(df[0], errors='coerce')
    df[1] = pd.to_numeric(df[1], errors='coerce')

    # 4. Delete the NaNs
    df = df.dropna()

    # 5. Safety check so we don't get a blind error!
    if len(df) == 0:
        print(f"🚨 WARNING: {filepath} is completely empty after cleaning. Please check the raw file formatting!")

    return df.to_numpy()

# ==========================================
# 2. SAFE INTERPOLATION FUNCTION
# ==========================================
def align_to_shared_grid(x_raw, y_raw, shared_x):
    """
    Resamples spectrum. 'fill_value="extrapolate"' prevents crashes
    if the grid is a tiny microscopic fraction of a nanometer longer than the data.
    """
    interpolator = interp1d(x_raw, y_raw, kind='cubic', bounds_error=False, fill_value="extrapolate")
    return interpolator(shared_x)

# ==========================================
# 3. MAIN PLOTTING FUNCTION
# ==========================================

def plot_simulated_crosstalk(t1_data, t2_data, sim1_data, sim2_data, name1="Target 1", name2="Target 2"):
    # 1. Unpack Wavelengths (X) and Intensities (Y)
    t1_x, t1_y = t1_data[:, 0], t1_data[:, 1]
    t2_x, t2_y = t2_data[:, 0], t2_data[:, 1]
    sim1_x, sim1_y = sim1_data[:, 0], sim1_data[:, 1]
    sim2_x, sim2_y = sim2_data[:, 0], sim2_data[:, 1]

    # ==========================================
    # MIN-MAX NORMALIZATION
    # ==========================================
    # This forces every single curve to have a minimum of 0 and a peak of 1
    # so we can fairly compare their physical shapes and overlaps!

    peak_height = 1.0 # Change this to 0.3 if you want them to stop at 0.3!

    t1_y = ((t1_y - t1_y.min()) / (t1_y.max() - t1_y.min())) * peak_height
    t2_y = ((t2_y - t2_y.min()) / (t2_y.max() - t2_y.min())) * peak_height

    sim1_y = ((sim1_y - sim1_y.min()) / (sim1_y.max() - sim1_y.min())) * peak_height
    sim2_y = ((sim2_y - sim2_y.min()) / (sim2_y.max() - sim2_y.min())) * peak_height
    # ==========================================

    # Create the "Safe Zone" Shared Grid
    # We find the highest start point and the lowest end point across all 4 files
    safe_min = max(t1_x.min(), t2_x.min(), sim1_x.min(), sim2_x.min())
    safe_max = min(t1_x.max(), t2_x.max(), sim1_x.max(), sim2_x.max())
    shared_x = np.linspace(safe_min, safe_max, 200)

    # Synchronize all four curves onto this exact same 200-point grid
    t1_aligned = align_to_shared_grid(t1_x, t1_y, shared_x)
    t2_aligned = align_to_shared_grid(t2_x, t2_y, shared_x)
    sim1_aligned = align_to_shared_grid(sim1_x, sim1_y, shared_x)
    sim2_aligned = align_to_shared_grid(sim2_x, sim2_y, shared_x)

    # Calculate Overlaps
    match_11 = t1_aligned * sim1_aligned
    miss_21  = t2_aligned * sim1_aligned
    miss_12  = t1_aligned * sim2_aligned
    match_22 = t2_aligned * sim2_aligned

    sum_11, sum_21 = np.sum(match_11), np.sum(miss_21)
    sum_12, sum_22 = np.sum(miss_12), np.sum(match_22)

    # Plot the 2x2 Grid
    fig, ax = plt.subplots(2, 2, figsize=(14, 10), sharex=True, sharey=True)

    # --- ROW 1: Sim Filter 1 ---
    ax[0, 0].plot(shared_x, t1_aligned, 'k--', label=name1)
    ax[0, 0].plot(shared_x, sim1_aligned, 'b-', label='Predicted pentacene')
    ax[0, 0].fill_between(shared_x, match_11, color='green', alpha=0.3, label=f'Overlap: {sum_11:.1f}')
    ax[0, 0].set_title(f'MATCH: {name1} x Predicted pentacene', fontsize=12)
    ax[0, 0].legend()
    ax[0, 0].grid(True, alpha=0.4)

    ax[0, 1].plot(shared_x, t2_aligned, 'k--', label=name2)
    ax[0, 1].plot(shared_x, sim1_aligned, 'b-', label='Predicted pentacene')
    ax[0, 1].fill_between(shared_x, miss_21, color='red', alpha=0.3, label=f'Overlap: {sum_21:.1f}')
    ax[0, 1].set_title(f'CROSSTALK: {name2} x Predicted pentacene', fontsize=12)
    ax[0, 1].legend()
    ax[0, 1].grid(True, alpha=0.4)

    # --- ROW 2: Sim Filter 2 ---
    ax[1, 0].plot(shared_x, t1_aligned, 'k--', label=name1)
    ax[1, 0].plot(shared_x, sim2_aligned, 'm-', label='Predicted polystyrene')
    ax[1, 0].fill_between(shared_x, miss_12, color='red', alpha=0.3, label=f'Overlap: {sum_12:.1f}')
    ax[1, 0].set_title(f'CROSSTALK: {name1} x Predicted polystyrene', fontsize=12)
    ax[1, 0].set_xlabel('Wavelength (Physical Units)')
    ax[1, 0].legend()
    ax[1, 0].grid(True, alpha=0.4)

    ax[1, 1].plot(shared_x, t2_aligned, 'k--', label=name2)
    ax[1, 1].plot(shared_x, sim2_aligned, 'm-', label='Predicted polystyrene')
    ax[1, 1].fill_between(shared_x, match_22, color='green', alpha=0.3, label=f'Overlap: {sum_22:.1f}')
    ax[1, 1].set_title(f'MATCH: {name2} x Predicted polystyrene', fontsize=12)
    ax[1, 1].set_xlabel('Wavelength (Physical Units)')
    ax[1, 1].legend()
    ax[1, 1].grid(True, alpha=0.4)

    plt.tight_layout()
    plt.show()

# ==========================================
# 4. EXECUTE SCRIPT
# ==========================================

# NOTE: Paste your exact folder path below, making sure it ends with a slash (/)!
data_dir = "/content/drive/MyDrive/PolaritonicONN/TrainingDataset/" # Replace this!

t1_csv = load_and_force_math(f"{data_dir}Target_spectra_pentacene.csv")
t2_csv = load_and_force_math(f"{data_dir}Target_spectra_polystyrene_cutted-um.csv")

sim1_csv = load_and_force_math(f"{data_dir}Spectra_from_pentacene-predicted_geometry.csv")
sim2_csv = load_and_force_math(f"{data_dir}Spectra_from_polystyrene-predicted_geometry.csv")

# Generate the plot!
plot_simulated_crosstalk(t1_csv, t2_csv, sim1_csv, sim2_csv, "Pentacene", "Polystyrene")